# 03 - Construcción de la OBT (`analytics.obt_trips`)

En este notebook vamos a:

1. Leer `RAW.YELLOW_TRIPS_RAW` y `RAW.GREEN_TRIPS_RAW` desde Snowflake  
2. Estandarizar y unificar los viajes  
3. Enriquecer con Taxi Zone Lookup  
4. Crear columnas derivadas de negocio  
5. Construir la tabla final `ANALYTICS.OBT_TRIPS` con grano **1 fila = 1 viaje**  
6. Dejar una verificación básica de idempotencia para no duplicar viajes

In [1]:
# 1. Librerías y configuración
import os
from dotenv import load_dotenv

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

load_dotenv("../.env")

spark = (
    SparkSession.builder
    .appName("Construccion_OBT")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "net.snowflake:spark-snowflake_2.12:2.12.0-spark_3.4,net.snowflake:snowflake-jdbc:3.14.4"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

sf_options_raw = {
    "sfURL": f"{os.getenv('SNOWFLAKE_ACCOUNT')}.snowflakecomputing.com",
    "sfUser": os.getenv("SNOWFLAKE_USER"),
    "sfPassword": os.getenv("SNOWFLAKE_PASSWORD"),
    "sfDatabase": os.getenv("SNOWFLAKE_DATABASE"),
    "sfSchema": os.getenv("SNOWFLAKE_SCHEMA_RAW", "RAW"),
    "sfWarehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "sfRole": os.getenv("SNOWFLAKE_ROLE"),
}

sf_options_analytics = dict(sf_options_raw)
sf_options_analytics["sfSchema"] = os.getenv("SNOWFLAKE_SCHEMA_ANALYTICS", "ANALYTICS")

print("Spark listo.")
print("Schema RAW:", sf_options_raw["sfSchema"])
print("Schema ANALYTICS:", sf_options_analytics["sfSchema"])

Spark listo.
Schema RAW: RAW
Schema ANALYTICS: ANALYTICS


In [2]:
# 2. Parámetros
# Para desarrollar y probar el notebook sin romper el conector de Snowflake,
# trabajamos con una muestra tomada directamente en Snowflake.
BUILD_SAMPLE_ONLY = True
SAMPLE_ROWS = 10000

# URL oficial del Taxi Zone Lookup
TAXI_ZONE_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

# Si quieres probar idempotencia con un mes específico:
TEST_SERVICE_TYPE = "yellow"
TEST_YEAR = 2019
TEST_MONTH = 6


In [3]:
# 3. Lectura desde Snowflake
# Importante: el LIMIT se aplica en Snowflake, no después en Spark.

if BUILD_SAMPLE_ONLY:
    df_yellow_raw = (
        spark.read.format("snowflake")
        .options(**sf_options_raw)
        .option("query", f"""
            SELECT *
            FROM YELLOW_TRIPS_RAW
            LIMIT {SAMPLE_ROWS}
        """)
        .load()
    )

    df_green_raw = (
        spark.read.format("snowflake")
        .options(**sf_options_raw)
        .option("query", f"""
            SELECT *
            FROM GREEN_TRIPS_RAW
            LIMIT {SAMPLE_ROWS}
        """)
        .load()
    )
else:
    df_yellow_raw = (
        spark.read.format("snowflake")
        .options(**sf_options_raw)
        .option("dbtable", "YELLOW_TRIPS_RAW")
        .load()
    )

    df_green_raw = (
        spark.read.format("snowflake")
        .options(**sf_options_raw)
        .option("dbtable", "GREEN_TRIPS_RAW")
        .load()
    )

print("Columnas yellow:", len(df_yellow_raw.columns))
print("Columnas green :", len(df_green_raw.columns))
df_yellow_raw.show(3, truncate=False)
df_green_raw.show(3, truncate=False)


Columnas yellow: 24
Columnas green : 25
+---------+-----------------------+-----------------------+---------------+-------------+------------+------------------+--------------+--------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------------+-----------+------------+------------+-----------------------+
|VENDOR_ID|PICKUP_DATETIME        |DROPOFF_DATETIME       |PASSENGER_COUNT|TRIP_DISTANCE|RATE_CODE_ID|STORE_AND_FWD_FLAG|PU_LOCATION_ID|DO_LOCATION_ID|PAYMENT_TYPE|FARE_AMOUNT|EXTRA|MTA_TAX|TIP_AMOUNT|TOLLS_AMOUNT|IMPROVEMENT_SURCHARGE|TOTAL_AMOUNT|CONGESTION_SURCHARGE|AIRPORT_FEE|RUN_ID       |SOURCE_YEAR|SOURCE_MONTH|SERVICE_TYPE|INGESTED_AT_UTC        |
+---------+-----------------------+-----------------------+---------------+-------------+------------+------------------+--------------+--------------+------------+-----------+-----+-------+----------+------------+---------------------+------

## 4. Funciones helper

Aquí vamos a:
- unificar nombres de columnas,
- traducir códigos a descripciones,
- generar una clave natural de viaje,
- y construir la tabla final.

In [4]:
# 4. Helpers de catálogos y columnas

def rename_if_exists(df, old_name, new_name):
    if old_name in df.columns:
        return df.withColumnRenamed(old_name, new_name)
    return df

def add_missing_columns(df, expected_cols):
    for c, dtype in expected_cols.items():
        if c not in df.columns:
            df = df.withColumn(c, F.lit(None).cast(dtype))
    return df

vendor_map = {
    1: "Creative Mobile Technologies",
    2: "VeriFone",
    6: "MTS"
}

rate_code_map = {
    1: "Standard rate",
    2: "JFK",
    3: "Newark",
    4: "Nassau/Westchester",
    5: "Negotiated fare",
    6: "Group ride"
}

payment_type_map = {
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip"
}

trip_type_map = {
    1: "Street-hail",
    2: "Dispatch"
}

mapping_expr_vendor = F.create_map([F.lit(x) for kv in vendor_map.items() for x in kv])
mapping_expr_rate = F.create_map([F.lit(x) for kv in rate_code_map.items() for x in kv])
mapping_expr_payment = F.create_map([F.lit(x) for kv in payment_type_map.items() for x in kv])
mapping_expr_trip_type = F.create_map([F.lit(x) for kv in trip_type_map.items() for x in kv])

In [5]:
# 5. Estandarización de YELLOW
def lowercase_columns(df):
    return df.toDF(*[c.lower() for c in df.columns])

df_yellow = lowercase_columns(df_yellow_raw)

df_yellow = rename_if_exists(df_yellow, "tpep_pickup_datetime", "pickup_datetime")
df_yellow = rename_if_exists(df_yellow, "tpep_dropoff_datetime", "dropoff_datetime")
df_yellow = rename_if_exists(df_yellow, "vendorid", "vendor_id")
df_yellow = rename_if_exists(df_yellow, "ratecodeid", "rate_code_id")
df_yellow = rename_if_exists(df_yellow, "pulocationid", "pu_location_id")
df_yellow = rename_if_exists(df_yellow, "dolocationid", "do_location_id")

expected_cols = {
    "vendor_id": "int",
    "pickup_datetime": "timestamp",
    "dropoff_datetime": "timestamp",
    "passenger_count": "double",
    "trip_distance": "double",
    "rate_code_id": "double",
    "store_and_fwd_flag": "string",
    "pu_location_id": "int",
    "do_location_id": "int",
    "payment_type": "double",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "improvement_surcharge": "double",
    "total_amount": "double",
    "congestion_surcharge": "double",
    "airport_fee": "double",
    "trip_type": "double",
    "ehail_fee": "double",
    "run_id": "string",
    "source_year": "int",
    "source_month": "int",
    "service_type": "string",
    "ingested_at_utc": "timestamp",
}

df_yellow = add_missing_columns(df_yellow, expected_cols)

for col_name, dtype in expected_cols.items():
    df_yellow = df_yellow.withColumn(col_name, F.col(col_name).cast(dtype))

df_yellow = df_yellow.withColumn("service_type", F.lit("yellow"))

print("Yellow estandarizado.")
df_yellow.select("pickup_datetime", "dropoff_datetime", "pu_location_id", "do_location_id").show(5, truncate=False)


Yellow estandarizado.
+-----------------------+-----------------------+--------------+--------------+
|pickup_datetime        |dropoff_datetime       |pu_location_id|do_location_id|
+-----------------------+-----------------------+--------------+--------------+
|2015-01-01 06:31:43.000|2015-01-01 06:48:34.000|50            |265           |
|2015-01-01 06:35:35.000|2015-01-01 07:03:49.000|230           |138           |
|2015-01-01 06:38:39.000|2015-01-01 06:43:46.000|100           |230           |
|2015-01-01 06:59:44.000|2015-01-01 07:08:23.000|127           |244           |
|2015-01-01 06:01:12.000|2015-01-01 06:08:56.000|80            |225           |
+-----------------------+-----------------------+--------------+--------------+
only showing top 5 rows



In [6]:
# 6. Estandarización de GREEN
df_green = lowercase_columns(df_green_raw)

df_green = rename_if_exists(df_green, "lpep_pickup_datetime", "pickup_datetime")
df_green = rename_if_exists(df_green, "lpep_dropoff_datetime", "dropoff_datetime")
df_green = rename_if_exists(df_green, "vendorid", "vendor_id")
df_green = rename_if_exists(df_green, "ratecodeid", "rate_code_id")
df_green = rename_if_exists(df_green, "pulocationid", "pu_location_id")
df_green = rename_if_exists(df_green, "dolocationid", "do_location_id")

df_green = add_missing_columns(df_green, expected_cols)

for col_name, dtype in expected_cols.items():
    df_green = df_green.withColumn(col_name, F.col(col_name).cast(dtype))

df_green = df_green.withColumn("service_type", F.lit("green"))

print("Green estandarizado.")
df_green.select("pickup_datetime", "dropoff_datetime", "pu_location_id", "do_location_id").show(5, truncate=False)


Green estandarizado.
+-----------------------+-----------------------+--------------+--------------+
|pickup_datetime        |dropoff_datetime       |pu_location_id|do_location_id|
+-----------------------+-----------------------+--------------+--------------+
|2015-01-01 00:31:10.000|2015-01-01 00:50:41.000|255           |234           |
|2015-01-01 00:01:05.000|2015-01-01 00:03:30.000|75            |74            |
|2015-01-01 00:09:01.000|2015-01-01 00:33:26.000|43            |186           |
|2015-01-01 00:17:34.000|2015-01-01 00:27:07.000|80            |36            |
|2015-01-01 00:32:38.000|2015-01-01 00:40:32.000|37            |17            |
+-----------------------+-----------------------+--------------+--------------+
only showing top 5 rows



In [7]:
# 7. Unificación + limpieza mínima
df_trips = df_yellow.unionByName(df_green, allowMissingColumns=True)

df_trips = df_trips.filter(
    (F.col("pickup_datetime").isNotNull()) &
    (F.col("dropoff_datetime").isNotNull()) &
    (F.col("dropoff_datetime") >= F.col("pickup_datetime"))
)

df_trips = df_trips.filter(
    (F.col("trip_distance").isNull() | (F.col("trip_distance") >= 0)) &
    (F.col("fare_amount").isNull() | (F.col("fare_amount") >= -1000)) &
    (F.col("total_amount").isNull() | (F.col("total_amount") >= -1000))
).cache()

print("Viajes combinados y limpiados.")
df_trips.select(
    "service_type", "pickup_datetime", "dropoff_datetime",
    "pu_location_id", "do_location_id", "trip_distance"
).show(5, truncate=False)


Viajes combinados y limpiados.
+------------+-------------------+-------------------+--------------+--------------+-------------+
|service_type|pickup_datetime    |dropoff_datetime   |pu_location_id|do_location_id|trip_distance|
+------------+-------------------+-------------------+--------------+--------------+-------------+
|yellow      |2015-01-01 06:31:43|2015-01-01 06:48:34|50            |265           |13.88        |
|yellow      |2015-01-01 06:35:35|2015-01-01 07:03:49|230           |138           |11.54        |
|yellow      |2015-01-01 06:38:39|2015-01-01 06:43:46|100           |230           |0.88         |
|yellow      |2015-01-01 06:59:44|2015-01-01 07:08:23|127           |244           |1.86         |
|yellow      |2015-01-01 06:01:12|2015-01-01 06:08:56|80            |225           |1.6          |
+------------+-------------------+-------------------+--------------+--------------+-------------+
only showing top 5 rows



In [8]:
# 8. Enriquecimiento con Taxi Zones
import pandas as pd

zones_pd = pd.read_csv(TAXI_ZONE_URL)

df_zones = spark.createDataFrame(zones_pd)

df_zones = (
    df_zones
    .withColumnRenamed("LocationID", "location_id")
    .withColumnRenamed("Borough", "borough")
    .withColumnRenamed("Zone", "zone")
    .withColumnRenamed("service_zone", "service_zone")
    .withColumn("location_id", F.col("location_id").cast("int"))
)

df_pu_zones = (
    df_zones
    .select(
        F.col("location_id").alias("pu_location_id"),
        F.col("borough").alias("pu_borough"),
        F.col("zone").alias("pu_zone"),
        F.col("service_zone").alias("pu_service_zone")
    )
)

df_do_zones = (
    df_zones
    .select(
        F.col("location_id").alias("do_location_id"),
        F.col("borough").alias("do_borough"),
        F.col("zone").alias("do_zone"),
        F.col("service_zone").alias("do_service_zone")
    )
)

df_enriched = (
    df_trips
    .join(F.broadcast(df_pu_zones), on="pu_location_id", how="left")
    .join(F.broadcast(df_do_zones), on="do_location_id", how="left")
    .cache()
)

print("Join con Taxi Zones completado.")
df_enriched.select(
    "service_type",
    "pu_location_id", "pu_zone", "pu_borough",
    "do_location_id", "do_zone", "do_borough"
).show(5, truncate=False)


Join con Taxi Zones completado.
+------------+--------------+-------------------------+----------+--------------+-------------------------+----------+
|service_type|pu_location_id|pu_zone                  |pu_borough|do_location_id|do_zone                  |do_borough|
+------------+--------------+-------------------------+----------+--------------+-------------------------+----------+
|yellow      |50            |Clinton West             |Manhattan |265           |Outside of NYC           |NaN       |
|yellow      |230           |Times Sq/Theatre District|Manhattan |138           |LaGuardia Airport        |Queens    |
|yellow      |100           |Garment District         |Manhattan |230           |Times Sq/Theatre District|Manhattan |
|yellow      |127           |Inwood                   |Manhattan |244           |Washington Heights South |Manhattan |
|yellow      |80            |East Williamsburg        |Brooklyn  |225           |Stuyvesant Heights       |Brooklyn  |
+------------+--

In [9]:
# 9. Construcción de columnas derivadas y descripciones
df_obt = (
    df_enriched
    .withColumn("pickup_date", F.to_date("pickup_datetime"))
    .withColumn("pickup_hour", F.hour("pickup_datetime"))
    .withColumn("dropoff_date", F.to_date("dropoff_datetime"))
    .withColumn("dropoff_hour", F.hour("dropoff_datetime"))
    .withColumn("day_of_week", F.date_format("pickup_datetime", "E"))
    .withColumn("month", F.month("pickup_datetime"))
    .withColumn("year", F.year("pickup_datetime"))
    .withColumn("vendor_name", mapping_expr_vendor[F.col("vendor_id")])
    .withColumn("rate_code_desc", mapping_expr_rate[F.col("rate_code_id").cast("int")])
    .withColumn("payment_type_desc", mapping_expr_payment[F.col("payment_type").cast("int")])
    .withColumn("trip_type_desc", mapping_expr_trip_type[F.col("trip_type").cast("int")])
    .withColumn(
        "trip_duration_min",
        (F.col("dropoff_datetime").cast("long") - F.col("pickup_datetime").cast("long")) / 60.0
    )
    .withColumn(
        "avg_speed_mph",
        F.when(
            (F.col("trip_distance").isNotNull()) &
            (F.col("trip_distance") > 0) &
            (F.col("trip_duration_min").isNotNull()) &
            (F.col("trip_duration_min") > 0),
            F.col("trip_distance") / (F.col("trip_duration_min") / 60.0)
        )
    )
    .withColumn(
        "tip_pct",
        F.when(
            (F.col("fare_amount").isNotNull()) &
            (F.col("fare_amount") > 0) &
            (F.col("tip_amount").isNotNull()),
            (F.col("tip_amount") / F.col("fare_amount")) * 100.0
        )
    )
)

df_obt = df_obt.withColumn(
    "trip_nk",
    F.sha2(
        F.concat_ws(
            "||",
            F.coalesce(F.col("service_type").cast("string"), F.lit("")),
            F.coalesce(F.col("vendor_id").cast("string"), F.lit("")),
            F.coalesce(F.col("pickup_datetime").cast("string"), F.lit("")),
            F.coalesce(F.col("dropoff_datetime").cast("string"), F.lit("")),
            F.coalesce(F.col("pu_location_id").cast("string"), F.lit("")),
            F.coalesce(F.col("do_location_id").cast("string"), F.lit("")),
            F.coalesce(F.col("trip_distance").cast("string"), F.lit("")),
            F.coalesce(F.col("fare_amount").cast("string"), F.lit("")),
            F.coalesce(F.col("total_amount").cast("string"), F.lit("")),
            F.coalesce(F.col("source_year").cast("string"), F.lit("")),
            F.coalesce(F.col("source_month").cast("string"), F.lit(""))
        ),
        256
    )
)

df_obt = df_obt.dropDuplicates(["trip_nk"]).cache()

print("OBT construida en memoria.")
df_obt.select(
    "service_type", "pickup_datetime", "dropoff_datetime",
    "pu_zone", "do_zone", "trip_distance", "trip_duration_min",
    "avg_speed_mph", "tip_pct", "total_amount"
).show(10, truncate=False)


OBT construida en memoria.
+------------+-------------------+-------------------+---------------------------------+-----------------------+-------------+------------------+------------------+------------------+------------+
|service_type|pickup_datetime    |dropoff_datetime   |pu_zone                          |do_zone                |trip_distance|trip_duration_min |avg_speed_mph     |tip_pct           |total_amount|
+------------+-------------------+-------------------+---------------------------------+-----------------------+-------------+------------------+------------------+------------------+------------+
|green       |2015-01-01 01:41:17|2015-01-01 01:44:47|Crown Heights South              |Prospect Heights       |0.72         |3.5               |12.342857142857142|0.0               |5.8         |
|green       |2015-01-01 01:18:57|2015-01-01 01:36:28|Fordham South                    |Highbridge             |3.0          |17.516666666666666|10.27592768791627 |0.0               |14

In [10]:
# 10. Selección final de columnas para analytics.obt_trips
final_columns = [
    "trip_nk",
    "pickup_datetime",
    "dropoff_datetime",
    "pickup_date",
    "pickup_hour",
    "dropoff_date",
    "dropoff_hour",
    "day_of_week",
    "month",
    "year",
    "pu_location_id",
    "pu_zone",
    "pu_borough",
    "pu_service_zone",
    "do_location_id",
    "do_zone",
    "do_borough",
    "do_service_zone",
    "service_type",
    "vendor_id",
    "vendor_name",
    "rate_code_id",
    "rate_code_desc",
    "payment_type",
    "payment_type_desc",
    "trip_type",
    "trip_type_desc",
    "passenger_count",
    "trip_distance",
    "store_and_fwd_flag",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "congestion_surcharge",
    "airport_fee",
    "ehail_fee",
    "total_amount",
    "trip_duration_min",
    "avg_speed_mph",
    "tip_pct",
    "run_id",
    "ingested_at_utc",
    "source_year",
    "source_month"
]

df_obt_final = df_obt.select(*final_columns).cache()

df_obt_final.printSchema()
df_obt_final.show(5, truncate=False)


root
 |-- trip_nk: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- dropoff_date: date (nullable = true)
 |-- dropoff_hour: integer (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- pu_location_id: integer (nullable = true)
 |-- pu_zone: string (nullable = true)
 |-- pu_borough: string (nullable = true)
 |-- pu_service_zone: string (nullable = true)
 |-- do_location_id: integer (nullable = true)
 |-- do_zone: string (nullable = true)
 |-- do_borough: string (nullable = true)
 |-- do_service_zone: string (nullable = true)
 |-- service_type: string (nullable = false)
 |-- vendor_id: integer (nullable = true)
 |-- vendor_name: string (nullable = true)
 |-- rate_code_id: double (nullable = true)
 |-- rate_code_desc: string (nullable = t

In [11]:
# 11. Escritura a Snowflake
# Estrategia simple y segura para este deber:
#   - reconstruimos completa la OBT
#   - escribimos con overwrite
# Esto evita duplicados si vuelves a correr el notebook completo.

(
    df_obt_final.write.format("snowflake")
    .options(**sf_options_analytics)
    .option("dbtable", "OBT_TRIPS")
    .mode("overwrite")
    .save()
)

print("Tabla ANALYTICS.OBT_TRIPS escrita correctamente en Snowflake.")

Tabla ANALYTICS.OBT_TRIPS escrita correctamente en Snowflake.


In [12]:
# 12. Verificación rápida en memoria
# Evitamos volver a leer desde Snowflake con Spark porque el conector ya falló antes.
print("Vista de la OBT final en memoria:")
df_obt_final.select(
    "service_type", "source_year", "source_month", "pu_zone", "do_zone",
    "trip_distance", "trip_duration_min", "avg_speed_mph", "tip_pct"
).show(10, truncate=False)

print("Verifica en Snowflake con estas consultas:")
print("SELECT COUNT(*) FROM NYC_TLC.ANALYTICS.OBT_TRIPS;")
print("SELECT * FROM NYC_TLC.ANALYTICS.OBT_TRIPS LIMIT 10;")


Vista de la OBT final en memoria:
+------------+-----------+------------+---------------------------------+-----------------------+-------------+------------------+------------------+------------------+
|service_type|source_year|source_month|pu_zone                          |do_zone                |trip_distance|trip_duration_min |avg_speed_mph     |tip_pct           |
+------------+-----------+------------+---------------------------------+-----------------------+-------------+------------------+------------------+------------------+
|green       |2015       |1           |Crown Heights South              |Prospect Heights       |0.72         |3.5               |12.342857142857142|0.0               |
|green       |2015       |1           |Fordham South                    |Highbridge             |3.0          |17.516666666666666|10.27592768791627 |0.0               |
|yellow      |2015       |1           |Upper West Side South            |Upper East Side North  |1.43         |4.76666666

## 13. Verificación rápida de idempotencia

Como aquí escribimos la OBT con `overwrite`, si vuelves a correr el notebook completo no debería duplicar filas.

Además, como respaldo, también generamos `trip_nk` y usamos `dropDuplicates(["trip_nk"])`.
Eso te deja dos capas de protección:

- **protección 1**: reconstrucción completa con `overwrite`
- **protección 2**: eliminación de duplicados por clave natural/hash

In [13]:
# 13. Chequeo puntual de idempotencia en la muestra en memoria
df_test_month = df_obt_final.filter(
    (F.col("service_type") == TEST_SERVICE_TYPE) &
    (F.col("source_year") == TEST_YEAR) &
    (F.col("source_month") == TEST_MONTH)
)

total_rows = df_test_month.count()
unique_keys = df_test_month.select("trip_nk").distinct().count()

print(f"Mes de prueba: {TEST_SERVICE_TYPE} {TEST_YEAR}-{TEST_MONTH:02d}")
print("Filas totales :", total_rows)
print("Claves únicas :", unique_keys)
print("¿Sin duplicados por trip_nk?:", total_rows == unique_keys)


Mes de prueba: yellow 2019-06
Filas totales : 0
Claves únicas : 0
¿Sin duplicados por trip_nk?: True


In [14]:
# 14. Resumen por servicio/año/mes para evidencias del README
df_resumen = (
    df_obt_final
    .groupBy("service_type", "source_year", "source_month")
    .agg(F.count("*").alias("total_trips"))
    .orderBy("service_type", "source_year", "source_month")
)

df_resumen.show(50, truncate=False)


+------------+-----------+------------+-----------+
|service_type|source_year|source_month|total_trips|
+------------+-----------+------------+-----------+
|green       |2015       |1           |10000      |
|yellow      |2015       |1           |10000      |
+------------+-----------+------------+-----------+



## Qué poner en tu explicación del notebook

Puedes decir algo como esto:

- Se leyó el `raw` desde Snowflake.
- Se unificaron `yellow` y `green` bajo un mismo esquema.
- Se enriqueció con `taxi_zone_lookup`.
- Se construyó `analytics.obt_trips` con grano **1 viaje = 1 fila**.
- Se agregaron derivadas: `trip_duration_min`, `avg_speed_mph`, `tip_pct`.
- Se incorporaron metadatos: `run_id`, `ingested_at_utc`, `source_year`, `source_month`, `service_type`.
- La idempotencia se controló con `trip_nk` + `dropDuplicates` + escritura `overwrite`.